# BOLA ABR Streaming

Back Bone Code Originally Sourced from: https://github.com/suwen02/BOLA-ABR-Streaming.git

## Original Code from Github Repo

### Original Functions

studentcode.py

In [ ]:
# BOLA (Buffer Occupancy based Lyapunov Algorithm)
# Based on: "BOLA: Near-Optimal Bitrate Adaptation for Online Videos"
# K. Spiteri, R. Urgaonkar, R. K. Sitaraman, IEEE/ACM Transactions on Networking, vol. 28, no. 4, pp. 1702–1715, Aug. 2020

import math

# Tunable parameters
Q_MAX_SECONDS = 22.0
TAU_MULTIPLIER = 1.3
GAMMA_RATIO = 0.5

V = None              # Lyapunov parameter
gamma_p = None        # Utility offset parameter
utilities = {}        # Pre-computed utility values
initialized = False   # Initialization flag


def student_entrypoint(Measured_Bandwidth, Previous_Throughput, Buffer_Occupancy,
                       Available_Bitrates, Video_Time, Chunk, Rebuffering_Time,
                       Preferred_Bitrate):
    # ABR entrypoint invoked by the simulator each time a chunk is requested
    global initialized

    # Convert string keys to int (JSON deserializes dict keys as strings)
    Available_Bitrates = {int(k): v for k, v in Available_Bitrates.items()}

    # Lazily initialize and choose bitrate using BOLA-BASIC
    if not initialized:
        initialize_bola(Buffer_Occupancy, Chunk, Available_Bitrates)

    buffer_level = Buffer_Occupancy['time']
    bitrates = sorted(Available_Bitrates.keys())
    return bola_select(buffer_level, bitrates, Available_Bitrates, Chunk)


def tau_floor(chunk_time):
    # Compute reservoir duration based on chunk time and multiplier
    scaled = chunk_time * TAU_MULTIPLIER
    return max(chunk_time, scaled)


def initialize_bola(buffer_occ, chunk, bitrates_dict):
    # Initialize BOLA parameters (called once)
    global V, gamma_p, utilities, initialized

    bitrates = sorted(bitrates_dict.keys())
    R_min = bitrates[0]
    R_max = bitrates[-1]

    tau = tau_floor(chunk['time'])
    Q_max = max(tau, Q_MAX_SECONDS)

    utilities = {R: math.log(float(R) / float(R_min)) for R in bitrates}

    u_max = utilities[R_max]
    u_min = utilities[R_min]  # Always 0

    # Set gamma_p (offset parameter for well-defined objective)
    gamma_p = GAMMA_RATIO * (u_max - u_min)

    V = (Q_max - tau) / (u_max + gamma_p) if (u_max + gamma_p) > 0 else 3.0

    initialized = True


def bola_select(buffer_level, bitrates, bitrate_dict, chunk):
    # Select bitrate based on buffer occupancy (BOLA-BASIC)
    global V, gamma_p, utilities

    tau = tau_floor(chunk['time'])  # Reservoir (minimum safe buffer)
    R_min = bitrates[0]

    if buffer_level <= tau:
        return R_min

    max_obj = float('-inf')
    selected = R_min

    for R in bitrates:
        u = utilities[R]  # Pre-computed utility

        S = math.sqrt(float(R) / float(R_min))

        Q_eff = buffer_level - tau

        obj = (V * (u + gamma_p) - Q_eff) / S

        if obj > max_obj:
            max_obj = obj
            selected = R

    selected = safety_check(selected, buffer_level, bitrates, tau)# Ensure safety by avoiding highest bitrate under low buffer conditions

    return selected


def safety_check(selected, buffer_level, bitrates, tau):# Ensure safety by avoiding highest bitrate under low buffer conditions
    if buffer_level < tau * 1.5:
        idx = bitrates.index(selected)
        if idx == len(bitrates) - 1 and len(bitrates) > 1:
            selected = bitrates[idx - 1]

    elif buffer_level < tau * 2:
        idx = bitrates.index(selected)
        if idx == len(bitrates) - 1 and buffer_level < tau * 3:
            if idx > 0:
                selected = bitrates[idx - 1]

    return selected

studentComm.py

In [ ]:
import socket
import json
import studentcode as studentcode

# Quick-and-dirty TCP Server:
# ss = socket.socket(socket.AF_INET, socket.SOCK_STREAM | socket.SOCK_CLOEXEC)
ss = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
ss.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
ss.bind(('localhost', 6000))
ss.listen(10)
print('Waiting for simulator')

(clientsocket, address) = ss.accept()


def recv_commands():
    message = ""


    while(1):
        messagepart = clientsocket.recv(2048).decode()

        message += messagepart
        if message[-1] == '\n':


            jsonargs = json.loads(messagepart)
            message = ""

            if(jsonargs["exit"] != 0):
                return

            #todo: json data sanitization
            bitrate = studentcode.student_entrypoint(jsonargs["Measured Bandwidth"], jsonargs["Previous Throughput"], jsonargs["Buffer Occupancy"], jsonargs["Available Bitrates"], jsonargs["Video Time"], jsonargs["Chunk"], jsonargs["Rebuffering Time"], jsonargs["Preferred Bitrate"])

            payload = json.dumps({"bitrate" : bitrate}) + '\n'
            clientsocket.sendall(payload.encode())







if __name__ == "__main__":

    recv_commands()
    ss.close()

simulator.py

In [ ]:
import sys
import json
from Classes import SimBuffer, NetworkTrace, Scorecard, simulator_comm

#this file was written by Zach Peats
#This is the video download and playback simulator for an ABR algorithm lab.
#The program simulates a video stream over a network, using a network trace and a video manifest
#for more information regarding usage and file specifications, check out the readme



verbose = False


def loadtrace(tracefile):

    with open(tracefile, 'r',encoding='utf-8') as infile:
        lines = infile.readlines()

    tracelog = []

    for line in lines:
        splitline = line.split(' ')
        if len(splitline) > 1:
            try:
                tracelog.append((float(splitline[0]), float(splitline[1])))

            except ValueError as e:
                print("Your trace file is poorly formed!")



    trace = NetworkTrace.NetworkTrace(tracelog)

    return trace


def loadmanifest(manifestfile):

    with open(manifestfile, 'r', encoding='utf-8') as infile:
        lines = infile.read()

    manifest = json.loads(lines)
    return manifest

def prep_bitrates(available_rates, chunk):
    rates = dict(map(lambda x, y: (x, y), available_rates, chunk))
    return rates

def prep_chunk(chunks_rem, manifest, chunk_num):
    params = {  "left" : chunks_remaining,
                "time" : manifest["Chunk_Time"],
                "current" : chunk_num
                }
    return params


if __name__ == "__main__":

    #check arguments for relevant flags

    if "-v" in sys.argv or "--verbose" in sys.argv:
        verbose = True


    #Load in network trace from input file

    trace = loadtrace(sys.argv[1])

    #read video manifest

    manifest = loadmanifest(sys.argv[2])

    #create scorecard for logging

    logger = Scorecard.Scorecard(1, 1, 1)

    #simulator setup

    buffer = SimBuffer.SimBuffer(manifest["Buffer_Size"])

    chunks_remaining = manifest["Chunk_Count"]
    current_time = 0
    prev_throughput = 0
    rebuff_time = 0
    pref_bitrate = manifest["Preferred_Bitrate"]

    stu_chunk_size = None

    chunk_list = [(key, value) for key, value in manifest["Chunks"].items()]

    chunk_iter = chunk_list.__iter__()

    #Communication loop with student (for all chunks):

    chunknum, chunk = next(chunk_iter, None)

    while chunk:
        #calculate and pack info to be sent to student
        # todo ensure input types are correct
        m_band = trace.get_current_timesegment(current_time)[1]
        buf_occ = buffer.get_student_params()
        av_bitrates = prep_bitrates(manifest["Available_Bitrates"],chunk)
        chunk_arg = prep_chunk(chunks_remaining, manifest, chunknum)





        #send info to student, get response
        chosen_bitrate = simulator_comm.send_req_json(m_band, prev_throughput, buf_occ, av_bitrates, current_time, chunk_arg, rebuff_time, pref_bitrate)


        #bad response checking, ensure chunk fits in buffer
        try:
            stu_chunk_size = av_bitrates[int(chosen_bitrate)]
        except( KeyError ):
            print("Student returned invalid bitrate, exiting")
            break

        if stu_chunk_size > buffer.available_space():
            #chunk chosen does not fit in buffer, wait .5s and resend request
            buffer_time = buffer.burn_time(.5)
            current_time += .5
            continue


        logger.log_bitrate_choice(current_time, chunknum, (chosen_bitrate, stu_chunk_size))

        #simulate download and playback
        time_elapsed = trace.simulate_download_from_time(current_time, stu_chunk_size)

        #round time to remove floating point errors
        #todo: this did not fix them
        time_elapsed = round(time_elapsed, 3)

        rebuff_time = buffer.sim_chunk_download(stu_chunk_size, chunk_arg["time"], time_elapsed)

        #update state variables
        prev_throughput = (stu_chunk_size * 8) / time_elapsed
        current_time += time_elapsed
        chunks_remaining -= 1

        logger.log_rebuffer(current_time - rebuff_time, rebuff_time)

        #log actions



        #get next chunk
        chunknum, chunk = next(chunk_iter, (None, None))


    #cleanup and return
    simulator_comm.send_exit()

    if(verbose):
        logger.output_verbose()
    else:
        logger.output_results()

rand_sizes.py

In [ ]:
import numpy as np
import json

CHUNK_TIME = 2
NUM_OF_CHUNKS = 30
low_BR = 500000
med_BR = 1000000
high_BR = 5000000
Buffer_Size = 40000000

# generate random chunk sizes for the manifest, WIP
low = np.random.normal(low_BR / 8, 5000 / 8, NUM_OF_CHUNKS) * CHUNK_TIME
med = np.random.normal(med_BR / 8, 10000 / 8, NUM_OF_CHUNKS) * CHUNK_TIME
high = np.random.normal(high_BR / 8, 50000 / 8, NUM_OF_CHUNKS) * CHUNK_TIME

data = {}
data['Video_Time'] = NUM_OF_CHUNKS * CHUNK_TIME
data['Chunk_Count'] = NUM_OF_CHUNKS
data['Chunk_Time'] = CHUNK_TIME
data['Buffer_Size'] = Buffer_Size
data['Available_Bitrates'] = [low_BR, med_BR, high_BR]
data['Chunks'] = {}

for i in range(30):
    data['Chunks'][str(i)] = [
        int(low[i]),
        int(med[i]),
        int(high[i])
    ]

print(json.dumps(data, indent=4))

grader.py

In [ ]:
import os
import time
import subprocess
import threading
import sys

#This file was written by Zach Peats
#This program repeatedly invokes Simulator.py and studentComm.py in order to run multiple testcases autonomously
#For more information check out the readme



def run_student_code():
    subprocess.run(['python', 'studentComm.py'])
    return


if __name__ == "__main__":

    #check for verbosity
    verboseflag = ""
    if "-v" in sys.argv or "--verbose" in sys.argv:
        verboseflag = "-v"

    switch_ratio = 1
    buffer_ratio = 1



    manifestfilename = 'manifest.json'
    tracefilename = 'trace.txt'



    outtext = []

    #iterate over all test directories
    for testdir in os.listdir('./tests/'):


        testpath = "./tests/" + testdir + "/"

        manifestpath = testpath + manifestfilename
        tracepath = testpath + tracefilename

        #Run student process
        student_thread = threading.Thread(target=run_student_code)
        student_thread.start()
        time.sleep(1)

        #run simulator process
        output = subprocess.run(['python', 'simulator.py', tracepath, manifestpath, verboseflag], capture_output=True )
        print(output)

        student_thread.join()

        #decode output and come up with a final score

        outputlines = output.stdout.decode('unicode_escape').split('\n')

        sanitizedoutput = [line.strip() for line in outputlines]

        outtext.append(testdir + ": ")
        outtext.append("\n")
        outtext += outputlines
        outtext.append("\n")

        average_bitrate = None
        buffer_time = None
        switches = None

        for line in sanitizedoutput:
            if "Average bitrate" in line:
                average_bitrate = float(line.split(':')[1])

            if "buffer time" in line:
                buffer_time = float(line.split(':')[1])

            if "switches" in line:
                switches = float(line.split(':')[1])

        #check to ensure that diagnostic scores are found
        if switches is not None and buffer_time is not None and average_bitrate is not None:
            buffer_penalty = pow( (1 - (.05 * buffer_ratio)), buffer_time)
            switch_penalty = pow( (1 - (.08 * switch_ratio)), switches)

            score = average_bitrate * buffer_penalty * switch_penalty

            outtext.append("Score:")
            outtext.append(str(score))
            outtext.append('\n')
            outtext.append('\n')
            outtext.append('\n')

        else:
            outtext.append("Unexpected output from the simulator")
            outtext.append('\n')
            outtext.append('\n')


    with open("grade.txt", 'w', encoding='utf-8') as outfile:
        outfile.writelines(outtext)

### Original Classes

NetworkTrace.py

In [ ]:
import math

#this file was written by Zach Peats
#A class used to simulate network traces

class NetworkTrace:


    def __init__(self, bandwidths):

        self.bwlist = bandwidths

    #returns the timesegment the time argument is within
    def get_current_timesegment(self, cur_time):
        return min(self.bwlist, key= lambda x: abs(x[0] - cur_time) if cur_time > x[0] else math.inf )


    def simulate_download_from_time(self, time, size):

        cum_time = 0
        timeseg = self.get_current_timesegment(time)

        while(1):
            #find when next bw change is
            next_set = None
            try:
                next_set = self.bwlist[ self.bwlist.index(timeseg) + 1 ]
            except( IndexError ):
                pass

            #if no next bw change, calculate the remaining time
            if not next_set:
                cum_time += size / (timeseg[1] / 8)
                return cum_time

            #find time remaining on current bw, drain download by corresponding amt
            down_time = next_set[0] - time
            cum_time += down_time
            size -= down_time * (timeseg[1] / 8)

            if size <= 0:
                #refund unused time
                unused_time = -1 * size / (timeseg[1] / 8)
                cum_time -= unused_time
                return cum_time

            timeseg = next_set
            time = timeseg[0]


Scorecard.py

In [ ]:

#this file was written by Zach Peats
#A class that logs chunk decisions, bitrate switches, and rebuffers

class Scorecard:

    def __init__(self, qual_coef, buf_coef, switch_coef):

        self.qual = qual_coef
        self.buf = buf_coef
        self.switch = switch_coef

        self.rebuffers = []
        self.switches = []
        self.chunk_info = []


    def log_bitrate_choice(self, time, chunknum, chunk):
        self.chunk_info.append({"number" : chunknum,
                                     "time" : time,
                                     "chunk" : chunk})

        self.switching_check()


    def log_rebuffer(self, time, buffer_length):
        if buffer_length == 0:
            return
        self.rebuffers.append((time,buffer_length))

    def switching_check(self):
        if len(self.chunk_info) > 1:
            if self.chunk_info[-1]["chunk"][0] != self.chunk_info[-2]["chunk"][0]:
                self.switches.append({"time" : self.chunk_info[-1]["time"],
                                      "prev_br" : self.chunk_info[-2]["chunk"][0],
                                      "post_br" : self.chunk_info[-1]["chunk"][0]})

    def get_buffer_time(self):
        total = 0
        for rebuffer in self.rebuffers:
            total += rebuffer[1]

        return total

    def get_avg_quality(self):
        total = 0
        for chunk in self.chunk_info:
            total += int(chunk["chunk"][0])

        avg = total / len(self.chunk_info)
        return avg

    def output_results(self):

        print("Results:")
        avg_quality = self.get_avg_quality()
        print("Average bitrate:" + str(avg_quality))

        total_time = self.get_buffer_time()
        print("buffer time:" + str(total_time))

        num_switches = len(self.switches)
        print("switches:" + str(num_switches))


    def output_verbose(self):

        print("Chunk info:")
        for chunk in self.chunk_info:
            print("number: " + str(chunk['number']) + ", time chosen :" + str(chunk["time"]) + ", bitrate: " + str(chunk["chunk"][0]) + ", actual size: " + str(chunk['chunk'][1]))

        print("")
        print("Buffer info:")
        for rebuffer in self.rebuffers:
            print("begin time: " + str(rebuffer[0]), ", length: " + str(rebuffer[1]))

        print("")
        print("Switch info:")
        for switch in self.switches:
            print("time chosen: " + str(switch["time"]) + ", previous bitrate: " + str(switch["prev_br"]) + ", new bitrate: "+ str(switch["post_br"]))

        print("")
        self.output_results()

SimBuffer.py

In [ ]:

#this file was written by Zach Peats
#A class that represents the video playback buffer.

class SimBuffer:

    def __init__(self, bufsize):
        self.size = bufsize
        self.chunks = [] #each chunk object is a tuple (size, chunk_time)
        self.time = 0
        self.cur_size = 0
        self.mid_chunk_time = 0


    def get_student_params(self):
        params = {}
        params["size"] = self.size
        params["current"] = self.cur_size
        params["time"] = self.time
        return params

    def available_space(self):
        #self.buffer_relative_time()
        return self.size - self.cur_size

    def sim_chunk_download(self, chunk_size, chunk_time, playback_time):
        if chunk_size > self.size - self.cur_size:
            print("Error: Chunk being added is too large to fit into buffer")
            return False

        buffer_time = self.sim_playback(playback_time)

        self.chunks.append((chunk_size,chunk_time))
        self.calculate_occupancy()
        self.calculate_time()
        return buffer_time


    def calculate_occupancy(self):
        self.cur_size = 0
        for chunk in self.chunks:
            self.cur_size += chunk[0]

    def burn_time(self, time):
        buffer_time = self.sim_playback(time)
        self.calculate_occupancy()
        self.calculate_time()
        return buffer_time

    def sim_playback(self, playback_time):

        while playback_time > 0:

            #if there are any chunks to be played
            if self.chunks:
                current_chunk = self.chunks.pop(0)

                chunk_time_remaining = current_chunk[1]

                playback_time -= chunk_time_remaining

                if playback_time < 0:
                    chunk_time_remaining = -1 * playback_time
                    self.chunks.insert(0, (current_chunk[0], chunk_time_remaining))
                    return 0


            #no chunks left to be played, buffering
            else:
                return playback_time

        #all playback was simulated, return 0 buffer time
        return 0

    def calculate_time(self):

        totaltime = 0
        for chunk in self.chunks:
            totaltime += chunk[1]

        self.time = totaltime
        return

simulator_comm.py

In [ ]:
import socket
import json
import time

# Cut-the-corners TCP Client:
# s = socket.socket(socket.AF_INET, socket.SOCK_STREAM | socket.SOCK_CLOEXEC)
s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
s.connect(('localhost', 6000))

def send_req_json(m_band, prev_throughput, buf_occ, av_bitrates, current_time, chunk_arg, rebuff_time, pref_bitrate ):

    #pack message
    req = json.dumps({"Measured Bandwidth" : m_band,
                     "Previous Throughput" : prev_throughput,
                     "Buffer Occupancy" : buf_occ,
                     "Available Bitrates" : av_bitrates,
                     "Video Time" : current_time,
                     "Chunk" : chunk_arg,
                     "Rebuffering Time" : rebuff_time,
                     "Preferred Bitrate" : pref_bitrate,
                     "exit" : 0})
    req += '\n'

    s.sendall(req.encode())

    message = ""
    while(1):
        messagepart = s.recv(2048).decode()

        #print(messagepart)
        message += messagepart
        if message[-1] == '\n':
            #print(message)

            response = json.loads(message)

            return response["bitrate"]


def send_exit():
    req = json.dumps({"exit" : 1})

    req += '\n'
    s.sendall(req.encode())


if __name__ == "__main__":
    pass

## Personal Implementation

## Input

### Starting

In [ ]:
import socket
import json

# Quick-and-dirty TCP Server:
# ss = socket.socket(socket.AF_INET, socket.SOCK_STREAM | socket.SOCK_CLOEXEC)
ss = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
ss.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
ss.bind(('localhost', 6000))
ss.listen(10)
print('Waiting for simulator')

(clientsocket, address) = ss.accept()


def recv_commands():
    message = ""


    while(1):
        messagepart = clientsocket.recv(2048).decode()

        message += messagepart
        if message[-1] == '\n':


            jsonargs = json.loads(messagepart)
            message = ""

            if(jsonargs["exit"] != 0):
                return

            #todo: json data sanitization
            bitrate = student_entrypoint(jsonargs["Measured Bandwidth"], jsonargs["Previous Throughput"], jsonargs["Buffer Occupancy"], jsonargs["Available Bitrates"], jsonargs["Video Time"], jsonargs["Chunk"], jsonargs["Rebuffering Time"], jsonargs["Preferred Bitrate"])

            payload = json.dumps({"bitrate" : bitrate}) + '\n'
            clientsocket.sendall(payload.encode())

if __name__ == "__main__":

    recv_commands()
    ss.close()

In [ ]:
import sys
import json

#this file was written by Zach Peats
#This is the video download and playback simulator for an ABR algorithm lab.
#The program simulates a video stream over a network, using a network trace and a video manifest
#for more information regarding usage and file specifications, check out the readme


verbose = False


def loadtrace(tracefile):

    with open(tracefile, 'r',encoding='utf-8') as infile:
        lines = infile.readlines()

    tracelog = []

    for line in lines:
        splitline = line.split(' ')
        if len(splitline) > 1:
            try:
                tracelog.append((float(splitline[0]), float(splitline[1])))

            except ValueError as e:
                print("Your trace file is poorly formed!")



    trace = NetworkTrace.NetworkTrace(tracelog)

    return trace


def loadmanifest(manifestfile):

    with open(manifestfile, 'r', encoding='utf-8') as infile:
        lines = infile.read()

    manifest = json.loads(lines)
    return manifest

def prep_bitrates(available_rates, chunk):
    rates = dict(map(lambda x, y: (x, y), available_rates, chunk))
    return rates

def prep_chunk(chunks_rem, manifest, chunk_num):
    params = {  "left" : chunks_remaining,
                "time" : manifest["Chunk_Time"],
                "current" : chunk_num
                }
    return params


if __name__ == "__main__":

    #check arguments for relevant flags

    if "-v" in sys.argv or "--verbose" in sys.argv:
        verbose = True


    #Load in network trace from input file

    trace = loadtrace(sys.argv[1])

    #read video manifest

    manifest = loadmanifest(sys.argv[2])

    #create scorecard for logging

    logger = Scorecard.Scorecard(1, 1, 1)

    #simulator setup

    buffer = SimBuffer.SimBuffer(manifest["Buffer_Size"])

    chunks_remaining = manifest["Chunk_Count"]
    current_time = 0
    prev_throughput = 0
    rebuff_time = 0
    pref_bitrate = manifest["Preferred_Bitrate"]

    stu_chunk_size = None

    chunk_list = [(key, value) for key, value in manifest["Chunks"].items()]

    chunk_iter = chunk_list.__iter__()

    #Communication loop with student (for all chunks):

    chunknum, chunk = next(chunk_iter, None)

    while chunk:
        #calculate and pack info to be sent to student
        # todo ensure input types are correct
        m_band = trace.get_current_timesegment(current_time)[1]
        buf_occ = buffer.get_student_params()
        av_bitrates = prep_bitrates(manifest["Available_Bitrates"],chunk)
        chunk_arg = prep_chunk(chunks_remaining, manifest, chunknum)

        #send info to student, get response
        chosen_bitrate = simulator_comm.send_req_json(m_band, prev_throughput, buf_occ, av_bitrates, current_time, chunk_arg, rebuff_time, pref_bitrate)


        #bad response checking, ensure chunk fits in buffer
        try:
            stu_chunk_size = av_bitrates[int(chosen_bitrate)]
        except( KeyError ):
            print("Student returned invalid bitrate, exiting")
            break

        if stu_chunk_size > buffer.available_space():
            #chunk chosen does not fit in buffer, wait .5s and resend request
            buffer_time = buffer.burn_time(.5)
            current_time += .5
            continue


        logger.log_bitrate_choice(current_time, chunknum, (chosen_bitrate, stu_chunk_size))

        #simulate download and playback
        time_elapsed = trace.simulate_download_from_time(current_time, stu_chunk_size)

        #round time to remove floating point errors
        #todo: this did not fix them
        time_elapsed = round(time_elapsed, 3)

        rebuff_time = buffer.sim_chunk_download(stu_chunk_size, chunk_arg["time"], time_elapsed)

        #update state variables
        prev_throughput = (stu_chunk_size * 8) / time_elapsed
        current_time += time_elapsed
        chunks_remaining -= 1

        logger.log_rebuffer(current_time - rebuff_time, rebuff_time)

        #log actions



        #get next chunk
        chunknum, chunk = next(chunk_iter, (None, None))


    #cleanup and return
    simulator_comm.send_exit()

    if(verbose):
        logger.output_verbose()
    else:
        logger.output_results()

## Data Transformation

## Output

## Validation

In [ ]:
import os
import time
import subprocess
import threading
import sys

#This file was written by Zach Peats
#This program repeatedly invokes Simulator.py and studentComm.py in order to run multiple testcases autonomously
#For more information check out the readme



def run_student_code():
    subprocess.run(['python', 'studentComm.py'])
    return


if __name__ == "__main__":

    #check for verbosity
    verboseflag = ""
    if "-v" in sys.argv or "--verbose" in sys.argv:
        verboseflag = "-v"

    switch_ratio = 1
    buffer_ratio = 1



    manifestfilename = 'manifest.json'
    tracefilename = 'trace.txt'



    outtext = []

    #iterate over all test directories
    for testdir in os.listdir('./tests/'):


        testpath = "./tests/" + testdir + "/"

        manifestpath = testpath + manifestfilename
        tracepath = testpath + tracefilename

        #Run student process
        student_thread = threading.Thread(target=run_student_code)
        student_thread.start()
        time.sleep(1)

        #run simulator process
        output = subprocess.run(['python', 'simulator.py', tracepath, manifestpath, verboseflag], capture_output=True )
        print(output)

        student_thread.join()

        #decode output and come up with a final score

        outputlines = output.stdout.decode('unicode_escape').split('\n')

        sanitizedoutput = [line.strip() for line in outputlines]

        outtext.append(testdir + ": ")
        outtext.append("\n")
        outtext += outputlines
        outtext.append("\n")

        average_bitrate = None
        buffer_time = None
        switches = None

        for line in sanitizedoutput:
            if "Average bitrate" in line:
                average_bitrate = float(line.split(':')[1])

            if "buffer time" in line:
                buffer_time = float(line.split(':')[1])

            if "switches" in line:
                switches = float(line.split(':')[1])

        #check to ensure that diagnostic scores are found
        if switches is not None and buffer_time is not None and average_bitrate is not None:
            buffer_penalty = pow( (1 - (.05 * buffer_ratio)), buffer_time)
            switch_penalty = pow( (1 - (.08 * switch_ratio)), switches)

            score = average_bitrate * buffer_penalty * switch_penalty

            outtext.append("Score:")
            outtext.append(str(score))
            outtext.append('\n')
            outtext.append('\n')
            outtext.append('\n')

        else:
            outtext.append("Unexpected output from the simulator")
            outtext.append('\n')
            outtext.append('\n')


    with open("grade.txt", 'w', encoding='utf-8') as outfile:
        outfile.writelines(outtext)